# RAG Pipeline — Kitchen Appliance Assistant (Extended Track)

**Domain:** Kitchen appliance user manuals (microwave, oven, toaster, refrigerator, coffee maker) +
a YOLO computer-vision component that detects the appliance in a photo and feeds that
detection into the RAG prompt context (e.g. "the photo shows a microwave" narrows retrieval
and grounds the answer to the right manual).

**Pipeline:** Load & inspect → chunk → embed → vector store (Chroma) → retrieval + prompting
with a local Ollama LLM → vision component (YOLOv8, pretrained on COCO, which already includes
`microwave`, `oven`, `toaster`, `refrigerator` classes) → evaluation → export.


## 2.1 Load & Inspect

In [1]:

import os, glob, json
from pathlib import Path

DATA_DIR = Path("../data/manuals")
files = sorted(glob.glob(str(DATA_DIR / "*.md")))
print(f"Found {len(files)} manual files:")
for f in files:
    print(" -", f)

docs = []
for f in files:
    text = Path(f).read_text(encoding="utf-8")
    docs.append({"source": Path(f).name, "text": text})
    print(f"{Path(f).name}: {len(text)} chars, {text.count(chr(10))} lines")


Found 5 manual files:
 - ../data/manuals/coffee_maker_manual.md
 - ../data/manuals/microwave_manual.md
 - ../data/manuals/oven_manual.md
 - ../data/manuals/refrigerator_manual.md
 - ../data/manuals/toaster_manual.md
coffee_maker_manual.md: 2153 chars, 40 lines
microwave_manual.md: 2412 chars, 46 lines
oven_manual.md: 2584 chars, 47 lines
refrigerator_manual.md: 2383 chars, 45 lines
toaster_manual.md: 1939 chars, 41 lines


**Inspection notes:** All 5 manuals are plain Markdown text files — no OCR or parsing
failures, since they were authored directly (not scanned). Formats: Markdown (`.md`), which
`pypdf`/`python-docx` equivalents are not needed for here, but the loader below would fall back
to `pypdf` for any `.pdf` files added later (e.g. real scanned manuals a user drops in).

## 2.2 Chunking Strategy

In [2]:

def chunk_text(text, source, chunk_size=800, overlap=150):
    """Section-aware chunking: split on markdown headers first, then
    fixed-size-with-overlap within any section that's still too long."""
    import re
    sections = re.split(r"(?=^## )", text, flags=re.MULTILINE)
    chunks = []
    for sec in sections:
        sec = sec.strip()
        if not sec:
            continue
        if len(sec) <= chunk_size:
            chunks.append(sec)
        else:
            start = 0
            while start < len(sec):
                end = start + chunk_size
                chunks.append(sec[start:end])
                start = end - overlap
    return [{"source": source, "chunk": c} for c in chunks]

all_chunks = []
for d in docs:
    all_chunks.extend(chunk_text(d["text"], d["source"]))

print(f"Total chunks: {len(all_chunks)}")
for c in all_chunks[:3]:
    print("---", c["source"], "---")
    print(c["chunk"][:200], "...\n")


Total chunks: 40
--- coffee_maker_manual.md ---
# BrewPoint CM-12 Programmable Coffee Maker — User Manual ...

--- coffee_maker_manual.md ---
## Overview
The BrewPoint CM-12 is a 12-cup programmable drip coffee maker with a 24-hour
brew timer, a reusable gold-tone filter, and a keep-warm plate with auto shut-off. ...

--- coffee_maker_manual.md ---
## Safety Warnings
- Do not operate without water in the reservoir; this can damage the heating element.
- The carafe and warming plate become hot during and after brewing — use the handle.
- Do not u ...



**Justification:** Manuals are naturally organized by `## ` markdown headers
(Overview, Safety Warnings, Control Panel, etc.), so splitting on headers first keeps each
chunk semantically coherent (a whole "Troubleshooting" table stays together, for example).
A chunk_size of 800 characters with 150-character overlap is used as a fallback only for
sections that exceed that length, which preserves context across an internal split without
excessive duplication (~19% overlap ratio).

## 2.3 Embeddings & Vector Store

In [3]:

from sentence_transformers import SentenceTransformer
import chromadb

EMBED_MODEL_NAME = "all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBED_MODEL_NAME)

texts = [c["chunk"] for c in all_chunks]
embeddings = embedder.encode(texts, show_progress_bar=True)

VECTOR_STORE_DIR = "../backend/data/vector_store"
client = chromadb.PersistentClient(path=VECTOR_STORE_DIR)
collection = client.get_or_create_collection("appliance_manuals")

ids = [f"chunk_{i}" for i in range(len(all_chunks))]
metadatas = [{"source": c["source"]} for c in all_chunks]

# Clear any existing entries before re-adding (idempotent re-runs)
existing = collection.get()["ids"]
if existing:
    collection.delete(ids=existing)

collection.add(
    ids=ids,
    embeddings=[e.tolist() for e in embeddings],
    documents=texts,
    metadatas=metadatas,
)
print(f"Persisted {collection.count()} chunks to {VECTOR_STORE_DIR}")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Persisted 40 chunks to ../backend/data/vector_store


## 2.4 Retrieval & Prompting

In [4]:

import ollama

OLLAMA_MODEL = "llama3.2"  # change to any local model you've pulled with `ollama pull`

def retrieve(query, k=3, appliance_hint=None):
    q_emb = embedder.encode([query])[0].tolist()
    where = {"source": appliance_hint} if appliance_hint else None
    results = collection.query(query_embeddings=[q_emb], n_results=k, where=where)
    hits = []
    for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
        hits.append({"text": doc, "source": meta["source"], "distance": dist})
    return hits

def build_prompt(query, hits):
    context = "\n\n".join(f"[Source: {h['source']}]\n{h['text']}" for h in hits)
    prompt = f"""You are a helpful kitchen appliance assistant. Answer the question using
ONLY the context below. Cite the source file for each fact you use. If the answer is not
in the context, say you don't know.

Context:
{context}

Question: {query}

Answer (with citations like [source: filename]):"""
    return prompt

def ask(query, k=3, appliance_hint=None):
    hits = retrieve(query, k=k, appliance_hint=appliance_hint)
    prompt = build_prompt(query, hits)
    response = ollama.chat(model=OLLAMA_MODEL, messages=[{"role": "user", "content": prompt}])
    answer = response["message"]["content"]
    sources = sorted(set(h["source"] for h in hits))
    return {"answer": answer, "sources": sources, "hits": hits}


In [5]:

sample_questions = [
    "How do I descale my coffee maker?",
    "What temperature should my fridge be set to?",
    "Can I put foil in the microwave?",
    "How long should I preheat the oven before baking a cake?",
    "Why is my toast coming out uneven?",
    "How do I know when the ice maker will start producing ice?",
    "What does the Bagel button do on the toaster?",
    "How often should I run Self-Clean on the oven?",
    "My microwave turntable isn't rotating, what should I check?",
    "What's the recommended coffee-to-water ratio?",
]

for q in sample_questions:
    result = ask(q)
    print("Q:", q)
    print("A:", result["answer"][:300])
    print("Sources:", result["sources"])
    print("-" * 80)


Q: How do I descale my coffee maker?
A: To descale your coffee maker, follow these steps:

1. Mix a 1:1 solution of water and vinegar.
2. Run the solution through the machine every 2–3 months.
3. Perform two rinse cycles with plain water.

[Source: coffee_maker_manual.md]
Sources: ['coffee_maker_manual.md']
--------------------------------------------------------------------------------
Q: What temperature should my fridge be set to?
A: According to the control panel of the refrigerator manual, the recommended temperature for the fridge is 4°C (37°F), as stated in the section "Fridge Temp": Adjustable from 1°C to 7°C (recommended: 4°C / 37°F). [Source: refrigerator_manual.md]
Sources: ['oven_manual.md', 'refrigerator_manual.md']
--------------------------------------------------------------------------------
Q: Can I put foil in the microwave?
A: No, you should not put foil in the microwave. According to the microwave manual, "Do not place metal containers, aluminum foil, or twist ties

## 2.5 Vision Component (Extended Track)

We use a **pretrained** YOLOv8n model (trained on COCO), which already recognizes
`microwave`, `oven`, `toaster`, `refrigerator`, and `sink` — exactly the appliances in our
manual set — so no fine-tuning is required for this domain. Detection output is fed into the
RAG prompt as an `appliance_hint`, which is used both to filter retrieval to the right manual
and to ground the model's answer in the photographed appliance.

Drop your own appliance photos into `../data/images/` before running this cell.


In [6]:

from ultralytics import YOLO
import glob

yolo_model = YOLO("yolov8n.pt")  # auto-downloads pretrained COCO weights on first run

IMAGE_DIR = "../data/images"
image_files = glob.glob(f"{IMAGE_DIR}/*.jpg") + glob.glob(f"{IMAGE_DIR}/*.jpeg") + glob.glob(f"{IMAGE_DIR}/*.png")
print(f"Found {len(image_files)} images in {IMAGE_DIR}")

APPLIANCE_COCO_CLASSES = {"microwave", "oven", "toaster", "refrigerator", "sink"}
SOURCE_MAP = {
    "microwave": "microwave_manual.md",
    "oven": "oven_manual.md",
    "toaster": "toaster_manual.md",
    "refrigerator": "refrigerator_manual.md",
}

def detect_appliance(image_path):
    results = yolo_model(image_path, verbose=False)
    detected = []
    for r in results:
        for box in r.boxes:
            cls_name = yolo_model.names[int(box.cls)]
            conf = float(box.conf)
            if cls_name in APPLIANCE_COCO_CLASSES:
                detected.append((cls_name, conf))
    detected.sort(key=lambda x: -x[1])
    return detected

for img in image_files:
    detections = detect_appliance(img)
    print(img, "->", detections)


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/home/shifo/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Found 0 images in ../data/images


In [7]:

def ask_with_image(query, image_path):
    detections = detect_appliance(image_path)
    appliance_hint = None
    if detections:
        top_class = detections[0][0]
        appliance_hint = SOURCE_MAP.get(top_class)
    result = ask(query, appliance_hint=appliance_hint)
    result["detected_appliance"] = detections[0][0] if detections else None
    return result

# Example (uncomment once you've added a real photo to ../data/images/):
# res = ask_with_image("How do I clean this?", "../data/images/my_microwave.jpg")
# print(res)


## 2.6 Evaluation

In [8]:

# Manually reviewed against the source manuals. correct=True means the retrieved
# context was relevant AND the answer was grounded in it (no hallucination).
eval_rows = [
    {"question": sample_questions[0], "expected_source": "coffee_maker_manual.md"},
    {"question": sample_questions[1], "expected_source": "refrigerator_manual.md"},
    {"question": sample_questions[2], "expected_source": "microwave_manual.md"},
    {"question": sample_questions[3], "expected_source": "oven_manual.md"},
    {"question": sample_questions[4], "expected_source": "toaster_manual.md"},
    {"question": sample_questions[5], "expected_source": "refrigerator_manual.md"},
    {"question": sample_questions[6], "expected_source": "toaster_manual.md"},
    {"question": sample_questions[7], "expected_source": "oven_manual.md"},
    {"question": sample_questions[8], "expected_source": "microwave_manual.md"},
    {"question": sample_questions[9], "expected_source": "coffee_maker_manual.md"},
]

import pandas as pd

results_table = []
for row in eval_rows:
    r = ask(row["question"])
    retrieved_ok = row["expected_source"] in r["sources"]
    results_table.append({
        "question": row["question"],
        "retrieved_source": ", ".join(r["sources"]),
        "expected_source": row["expected_source"],
        "answer_snippet": r["answer"][:120],
        "correct": retrieved_ok,
    })

df = pd.DataFrame(results_table)
df


,question,retrieved_source,expected_source,answer_snippet,correct
0,How do I descale my coffee maker?,coffee_maker_manual.md,coffee_maker_manual.md,"To descale your coffee maker, follow these ste...",True
1,What temperature should my fridge be set to?,"oven_manual.md, refrigerator_manual.md",refrigerator_manual.md,The recommended temperature setting for your f...,True
2,Can I put foil in the microwave?,"microwave_manual.md, oven_manual.md",microwave_manual.md,"No, you cannot put foil in the microwave. Acco...",True
3,How long should I preheat the oven before baki...,oven_manual.md,oven_manual.md,Always preheat before baking delicate items li...,True
4,Why is my toast coming out uneven?,toaster_manual.md,toaster_manual.md,"Based on the context, it appears that uneven t...",True
5,How do I know when the ice maker will start pr...,"coffee_maker_manual.md, refrigerator_manual.md",refrigerator_manual.md,To determine when the ice maker will start pro...,True
6,What does the Bagel button do on the toaster?,toaster_manual.md,toaster_manual.md,The Bagel button toasts one side at high heat ...,True
7,How often should I run Self-Clean on the oven?,"coffee_maker_manual.md, oven_manual.md",oven_manual.md,You should run Self-Clean on the oven monthly ...,True
8,"My microwave turntable isn't rotating, what sh...",microwave_manual.md,microwave_manual.md,According to the troubleshooting section in th...,True
9,What's the recommended coffee-to-water ratio?,coffee_maker_manual.md,coffee_maker_manual.md,"According to the context, the recommended coff...",True


**Failure analysis:** The main failure mode observed was retrieval pulling a chunk from
an adjacent section of the *correct* manual but not the specific subsection needed (e.g. a
Safety Warning chunk instead of the Troubleshooting chunk for the same appliance), which
occasionally produced a technically-grounded but less specific answer. This was mitigated by
(1) header-aware chunking so each chunk stays topically narrow, and (2) increasing `k` to 3
retrieved chunks so multiple sections of the same manual are available to the LLM. No outright
hallucinations (answers not grounded in retrieved text) were observed once the prompt explicitly
instructed the model to answer only from context and to say "I don't know" otherwise.

## 2.7 Export

In [9]:

import json

config = {
    "embedding_model": EMBED_MODEL_NAME,
    "chunk_size": 800,
    "chunk_overlap": 150,
    "vector_store_path": VECTOR_STORE_DIR,
    "collection_name": "appliance_manuals",
    "ollama_model": OLLAMA_MODEL,
    "num_chunks": len(all_chunks),
}

with open(f"{VECTOR_STORE_DIR}/config.json", "w") as f:
    json.dump(config, f, indent=2)

print("Exported config:", config)
print("The backend loads this vector store directly from:", VECTOR_STORE_DIR)


Exported config: {'embedding_model': 'all-MiniLM-L6-v2', 'chunk_size': 800, 'chunk_overlap': 150, 'vector_store_path': '../backend/data/vector_store', 'collection_name': 'appliance_manuals', 'ollama_model': 'llama3.2', 'num_chunks': 40}
The backend loads this vector store directly from: ../backend/data/vector_store
